In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import MessagesState, START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv
import os
load_dotenv()  # 必须要这句！
@tool
def play_song_on_qq(song: str):
    """在qq音乐上播放歌曲"""
    # 调用QQ音乐 API...
    return f"成功在QQ音乐上播放了{song}!"

@tool
def play_song_on_163(song: str):
    """在网易云上播放歌曲"""
    # 调用网易云 API...
    return f"成功在网易云上播放了{song}!"

tools = [play_song_on_qq, play_song_on_163]
tool_node = ToolNode(tools)

In [18]:
qwen = ChatOpenAI(
    model="qwen-plus",  # 或 qwen-max / qwen-turbo
    temperature=0,
    openai_api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
model = qwen.bind_tools(tools,parallel_tool_calls=False) #这个参数 parallel_tool_calls=False 是用来控制模型是否允许在单轮输出中同时调用多个工具的。简单来说，它是决定 AI 是“一次只做一件事”还是“一次做一堆事”的开关。




In [19]:
def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]
    # 如果没有函数调用，则结束
    if not last_message.tool_calls:
        return "end"
    # 否则如果有，我们继续
    else:
        return "continue"

# 定义调用模型的节点函数
def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    # 返回列表，因为这将被添加到现有列表中
    return {"messages": [response]}

In [20]:
# 定义一个新图
workflow = StateGraph(MessagesState)

# 定义我们将循环的两个节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 将入口点设置为 `agent`
# 这意味着这个节点是第一个被调用的
workflow.add_edge(START, "agent")

# 现在添加一个条件边
workflow.add_conditional_edges(
    # 首先，我们定义起始节点。我们使用 `agent`。
    # 这意味着这些是在调用 `agent` 节点后采取的边。
    "agent",
    # 接下来，我们传入将确定下一个调用哪个节点的函数。
    should_continue,
    # 最后我们传入一个映射。
    # 键是字符串，值是其他节点。
    # END是一个特殊节点，标记图应该结束。
    # 将会发生的是我们调用`should_continue`，然后该函数的输出
    # 将与此映射中的键匹配。
    # 根据匹配的键，然后调用相应的节点。
    {
        # 如果是`tools`，则调用工具节点。
        "continue": "action",
        # 否则我们结束。
        "end": END,
    },

)
# 现在我们从`tools`到`agent`添加一个普通边。
# 这意味着在调用`tools`之后，下一步调用`agent`节点。
workflow.add_edge("action", "agent")

# 设置内存
memory = MemorySaver()

# 最后，我们编译它！
# 这将它编译成一个LangChain Runnable，
# 意味着你可以像使用任何其他runnable一样使用它
# 我们添加`interrupt_before=["action"]`
# 这将在调用`action`节点之前添加一个断点
app = workflow.compile(checkpointer=memory)

In [21]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
input_message = HumanMessage(content="你能播放一首周杰伦播放量最高的歌曲吗？")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

你能播放一首周杰伦播放量最高的歌曲吗？
================================== Ai Message ==================================
Tool Calls:
  play_song_on_qq (call_601853cdd944493f964918)
 Call ID: call_601853cdd944493f964918
  Args:
    song: 周杰伦
================================= Tool Message =================================
Name: play_song_on_qq

成功在QQ音乐上播放了周杰伦!
================================== Ai Message ==================================

已在QQ音乐上成功播放周杰伦的歌曲！如果你有特定想听的歌曲，或者想切换到其他平台（如网易云音乐），随时告诉我哦！ 😊


检查记录并且重放

In [22]:
app.get_state(config).values["messages"]
all_state=[]

for state in app.get_state_history(config):
    print(state)
    human_contents = [m.content for m in state.values.get("messages", [])]
    print(human_contents)
    all_state.append(state)
    print("-----")


StateSnapshot(values={'messages': [HumanMessage(content='你能播放一首周杰伦播放量最高的歌曲吗？', additional_kwargs={}, response_metadata={}, id='d6dfb3a1-fadd-4292-8bae-26e962a77e00'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 222, 'total_tokens': 273, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-415e92ad-9876-960f-af3d-af62ec3c2570', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc8f4-bfe1-7b13-a2d6-cc264f8ba1bb-0', tool_calls=[{'name': 'play_song_on_qq', 'args': {'song': '周杰伦'}, 'id': 'call_601853cdd944493f964918', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 51, 'total_tokens': 273, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}), ToolMessage(content='成功在QQ

In [23]:
to_replay = all_state[2]
to_replay.values

{'messages': [HumanMessage(content='你能播放一首周杰伦播放量最高的歌曲吗？', additional_kwargs={}, response_metadata={}, id='d6dfb3a1-fadd-4292-8bae-26e962a77e00'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 222, 'total_tokens': 273, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-415e92ad-9876-960f-af3d-af62ec3c2570', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc8f4-bfe1-7b13-a2d6-cc264f8ba1bb-0', tool_calls=[{'name': 'play_song_on_qq', 'args': {'song': '周杰伦'}, 'id': 'call_601853cdd944493f964918', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 51, 'total_tokens': 273, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}

In [24]:
to_replay.next

('action',)

In [29]:
# 修改最后一个消息的工具调用 分叉
# 我们将其从 `play_song_on_qq` 更改为 `play_song_on_163`
last_message = to_replay.values["messages"][-1]
last_message.tool_calls[0]["name"] = "play_song_on_163"

branch_config = app.update_state(
    to_replay.config,
    {"messages": [last_message]},
)
for event in app.stream(None, branch_config):
    print(event)
    # for v in event.values():
    #     print(v)

{'action': {'messages': [ToolMessage(content='成功在网易云上播放了周杰伦!', name='play_song_on_163', id='dfe94a73-2925-4943-a166-6b6dc046c033', tool_call_id='call_601853cdd944493f964918')]}}
{'agent': {'messages': [AIMessage(content='已在网易云音乐上成功播放周杰伦的歌曲！如果你有特定想听的歌曲，或者想切换到QQ音乐播放，随时告诉我哦！ 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 272, 'total_tokens': 307, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-402ff9a4-923b-9f8c-a6a6-d8cf47bd1c74', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc8fd-ef7c-7010-a6fa-d26a78a2a9ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 272, 'output_tokens': 35, 'total_tokens': 307, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}})]}}
